In [1]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

# R0线性外推设置（沿用第二个文件）
R0_TARGET_AFTER_PAUSE_SEC = 0.5
R0_FIT_POINT_START = 2
R0_FIT_POINT_END = 6
FIRST_TWO_VOLTAGE_EQUAL_ATOL = 1e-9
ZERO_CURRENT_LIMIT = 1e-6
VOLTAGE_JUMP_LIMIT = 1e-3
MAX_PAUSE_TO_PULSE_GAP_SEC = 5.0
R0_EARLY_WINDOW_FLAG = "后段电流不稳定，R0仅使用早期稳定窗口"

TRUSTED_R0_QUALITY_FLAGS = {
    "正常",
    R0_EARLY_WINDOW_FLAG,
    "首点0且电压跳变"
}

R0_CONFIDENCE_FULL = "完全可信（权重1.0）"
R0_CONFIDENCE_REVIEW = "需复核（权重0.5）"
R0_CONFIDENCE_INVALID = "不可用（权重0.0）"

PULSE_OUTPUT_COLUMNS = OUTPUT_COLUMNS + [
    "R0",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence"
]


Note: you may need to restart the kernel to use updated packages.


In [2]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [3]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间、电流、电压处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")
    df_td["Voltage"] = pd.to_numeric(df_td["Voltage"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def get_effective_start_pos(group):
    # 返回pulse段内第一个非零有效电流点的位置。
        current_abs = group["Current"].abs().to_numpy()
        non_zero_pos = np.flatnonzero(current_abs > ZERO_CURRENT_LIMIT)

        if len(non_zero_pos) == 0:
            return None

        return int(non_zero_pos[0])

    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            return True

        # 如果首个测量点 Current == 0，则从第一个非零有效点开始判断稳定性
        current_values = group["Current"].iloc[effective_start_pos:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    # 不再因为pulse后段电流下降而提前删除整段数据。
    # R0是否可计算，改为在下面仅依据实际拟合窗口内的电流稳定性判断。

    # -----------------------------
    # 7. 计算R0，并且每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def add_quality(base_quality, new_quality):
        if base_quality == "正常":
            return new_quality
        return base_quality + "；" + new_quality

    def calculate_r0_for_segment(group, effective_start_pos):
        group = group.sort_values("Time")

        r0_result = {
            "R0": np.nan,
            "R0_Target_Time": pd.NaT,
            "Pause_to_Pulse_Time_Diff_s": np.nan,
            "R0_Quality": "正常"
        }

        file_name = group["File"].iloc[0]
        pulse_start_time = group["Time"].iloc[0]
        first_current = group["Current"].iloc[0]

        previous_pause = df_td[
            (df_td["File"] == file_name)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ].sort_values("Time").tail(1)

        if previous_pause.empty:
            r0_result["R0_Quality"] = "无法计算R0：无前置pause点"
            return r0_result

        pause_time = previous_pause["Time"].iloc[0]
        pause_voltage = previous_pause["Voltage"].iloc[0]

        pause_to_pulse_gap_s = (pulse_start_time - pause_time).total_seconds()
        r0_result["Pause_to_Pulse_Time_Diff_s"] = pause_to_pulse_gap_s

        target_time = pause_time + pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC)
        r0_result["R0_Target_Time"] = target_time

        # 大于5s：标记为无效，不再进行长距离反向外推
        if pause_to_pulse_gap_s > MAX_PAUSE_TO_PULSE_GAP_SEC:
            r0_result["R0_Quality"] = "pause结束点到pulse首点时间差>5s"
            return r0_result

        # pulse段第一个点为0时：无论voltage是否跳变，R0计算都从第一个非零有效点开始；
        # 若voltage已经跳变，则额外给质量标记。
        if abs(first_current) <= ZERO_CURRENT_LIMIT:
            first_voltage = group["Voltage"].iloc[0]

            if (
                pd.notna(first_voltage)
                and pd.notna(pause_voltage)
                and abs(first_voltage - pause_voltage) > VOLTAGE_JUMP_LIMIT
            ):
                r0_result["R0_Quality"] = "首点0且电压跳变"

        # 默认使用有效pulse第2-6点进行线性拟合。
        # 若有效pulse第1点与第2点的电压相同（可能是重复采样），
        # 则跳过前两点，改用第3-6点进行线性外推。
        fit_point_start = R0_FIT_POINT_START

        if effective_start_pos + 1 < len(group):
            first_pulse_voltage = group["Voltage"].iloc[effective_start_pos]
            second_pulse_voltage = group["Voltage"].iloc[effective_start_pos + 1]

            first_two_voltage_equal = (
                pd.notna(first_pulse_voltage)
                and pd.notna(second_pulse_voltage)
                and np.isclose(
                    float(first_pulse_voltage),
                    float(second_pulse_voltage),
                    rtol=0.0,
                    atol=FIRST_TWO_VOLTAGE_EQUAL_ATOL
                )
            )

            if first_two_voltage_equal:
                fit_point_start = 3

        fit_start_pos = effective_start_pos + fit_point_start - 1
        fit_end_pos = effective_start_pos + R0_FIT_POINT_END

        fit_points = group.iloc[fit_start_pos:fit_end_pos].dropna(
            subset=["Time", "Voltage", "Current"]
        )

        if len(fit_points) < 2:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：拟合点不足"
            )
            return r0_result

        # R0只要求实际参与2-6点（或3-6点）拟合的早期窗口电流稳定。
        fit_current_values = fit_points["Current"]
        fit_current_abs_level = round(fit_current_values.abs().iloc[0], 1)
        fit_current_std = fit_current_values.std()

        if fit_current_abs_level == 1.5:
            fit_current_unstable = fit_current_std > STD_LIMIT_1P5A
        elif fit_current_abs_level == 3.0:
            fit_current_unstable = fit_current_std > STD_LIMIT_3A
        else:
            fit_current_unstable = True

        if fit_current_unstable:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：早期拟合窗口电流不稳定"
            )
            return r0_result

        # 若整段电流不稳定、但早期拟合窗口稳定，仍计算R0并添加专用flag。
        if is_bad_current_segment(group):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                R0_EARLY_WINDOW_FLAG
            )

        effective_current = group["Current"].iloc[effective_start_pos]

        if abs(effective_current) <= ZERO_CURRENT_LIMIT:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：无非零有效电流"
            )
            return r0_result

        if pd.isna(pause_voltage):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：pause电压缺失"
            )
            return r0_result
        
        # 判断采样点距离pulse起点的距离
        x_sec = (fit_points["Time"] - target_time) / pd.Timedelta(seconds=1)

        y_voltage = fit_points["Voltage"].astype(float)

        slope, intercept = np.polyfit(x_sec.to_numpy(), y_voltage.to_numpy(), 1)
        extrapolated_voltage = intercept

        r0_result["R0"] = abs(
            (extrapolated_voltage - pause_voltage) / effective_current
        )

        return r0_result

    selected_indices = []
    r0_results = {}

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            continue

        # 保留原逻辑：每段用“有效pulse起点后的第2个测量点”记录；
        # 如果点数不够，则退回到有效pulse起点本身。
        record_pos = effective_start_pos + 1
        if record_pos >= len(group):
            record_pos = effective_start_pos

        record_index = group.index[record_pos]
        selected_indices.append(record_index)

        r0_results[record_index] = calculate_r0_for_segment(
            group,
            effective_start_pos
        )

    pulse_sequence = pulse_sequence.loc[selected_indices].copy()

    r0_result_columns = [
        "R0",
        "R0_Target_Time",
        "Pause_to_Pulse_Time_Diff_s",
        "R0_Quality"
    ]

    for column in r0_result_columns:
        pulse_sequence[column] = pulse_sequence.index.map(
            lambda idx, col=column: r0_results[idx][col]
        )

    pulse_sequence = pulse_sequence.reset_index(drop=True)

    # -----------------------------
    # 8. 生成逐条R0置信度
    # -----------------------------
    def classify_r0_confidence(row):
        r0_value = row["R0"]
        quality_text = str(row["R0_Quality"]).strip()

        # 没有有效R0时，无论带有什么flag，都不能参与R0分析。
        if pd.isna(r0_value) or not np.isfinite(r0_value):
            return R0_CONFIDENCE_INVALID

        flags = {
            flag.strip()
            for flag in quality_text.split("；")
            if flag.strip()
        }

        # 只要一条记录所含flag全部属于已验证的干净标签，就给权重1.0。
        if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
            return R0_CONFIDENCE_FULL

        # 将来若新增了仍可计算R0、但尚未验证的flag，先标记为需复核。
        return R0_CONFIDENCE_REVIEW

    pulse_sequence["R0_Confidence"] = pulse_sequence.apply(
        classify_r0_confidence,
        axis=1
    )

    # -----------------------------
    # 9. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[PULSE_OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence

pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

# 中间结果预览；统一的R0_Quality统计放在最终绘图cell中，避免重复输出。
display(pulse_sequence)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.760000+00:00,-1.499470,4.036492,DCH,12_17,DCH/-1.5,NaN,NaT,NaN,无法计算R0：无前置pause点,不可用（权重0.0）
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,0.025070,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,0.024941,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.310000+00:00,2.995714,4.165722,CHA,12_29,CHA/3.0,0.024550,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.640000+00:00,-1.498840,3.725950,DCH,12_35,DCH/-1.5,NaN,2024-11-12 15:36:43.330000+00:00,15553.57,pause结束点到pulse首点时间差>5s,不可用（权重0.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,0.017858,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
280,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.810000+00:00,-1.497851,3.192480,DCH,9_54,DCH/-1.5,NaN,2024-10-24 00:01:13.750000+00:00,15695.30,pause结束点到pulse首点时间差>5s,不可用（权重0.0）
281,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,0.019617,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）
282,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,0.019862,2024-10-24 06:24:14.600000+00:00,0.86,正常,完全可信（权重1.0）


In [4]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
# -----------------------------
def filter_pulse(df):

    pulse_sequence_filter = df[~df["Zustand/Current"].isin(["DCH/-1.5"])].copy()
    return pulse_sequence_filter

filtered_pulse = filter_pulse(pulse_sequence)
display(filtered_pulse)

,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,0.025070,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,0.024941,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.310000+00:00,2.995714,4.165722,CHA,12_29,CHA/3.0,0.024550,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,0.018330,2024-11-12 20:56:39.410000+00:00,0.81,正常,完全可信（权重1.0）
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,0.018422,2024-11-12 21:57:22.670000+00:00,0.85,正常,完全可信（权重1.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,0.017919,2024-10-23 22:29:48.980000+00:00,0.84,正常,完全可信（权重1.0）
279,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,0.017858,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
281,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,0.019617,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）
282,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,0.019862,2024-10-24 06:24:14.600000+00:00,0.86,正常,完全可信（权重1.0）


In [5]:
# -----------------------------
# 10. 使用线性外推得到的 R0 结果
# -----------------------------
r0_result = filtered_pulse.copy()

# 第二个文件中的 R0 以 Ohm 输出；转换为第一个文件二阶拟合所需的 mOhm。
r0_result["R0"] = pd.to_numeric(r0_result["R0"], errors="coerce") * 1000.0

display(r0_result)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,25.070093,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,24.941121,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.310000+00:00,2.995714,4.165722,CHA,12_29,CHA/3.0,24.550339,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,18.330317,2024-11-12 20:56:39.410000+00:00,0.81,正常,完全可信（权重1.0）
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,18.422246,2024-11-12 21:57:22.670000+00:00,0.85,正常,完全可信（权重1.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,17.918675,2024-10-23 22:29:48.980000+00:00,0.84,正常,完全可信（权重1.0）
279,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,17.858465,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
281,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,19.616687,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）
282,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,19.861875,2024-10-24 06:24:14.600000+00:00,0.86,正常,完全可信（权重1.0）


In [6]:
# =============================================================================
# 11. 二阶 RC 拟合：在已计算 R0 的基础上计算 R1 / R2
# =============================================================================
# 模型说明：
#   这里沿用原始数据的电流符号：CHA 为正，DCH 为负。
#   V = OCV + Current * R0 + V1 + V2
#   dV1/dt = -V1/tau1 + Current*R1/tau1
#   dV2/dt = -V2/tau2 + Current*R2/tau2
#
# 拟合窗口：
#   前一个 PAUO 末端点用于初始化 OCV 和 RC 状态；
#   当前 pulse 段的所有采样点参与拟合；
#   当前 pulse 后面的 PAUO 段所有采样点参与拟合。
#
# 输出：
#   R0 / R1 / R2 单位均为 mOhm
#   tau1 / tau2 单位为 s
#   C1 / C2 单位为 F
#
# 说明（本版改动）：
#   经过边界钳制排查，90% SOC / 10% SOC 这一版的 R1/R2 拟合存在系统性触界问题，
#   结果不可信；50% SOC 没有任何边界命中、拟合质量最好，因此本版本只保留 50% SOC
#   的二阶 RC 拟合逻辑，其余 SOC 不再计算 R1/R2（R0 计算部分不受影响，仍对所有 SOC 生效）。
# =============================================================================
RC2_INITIAL_GUESS_50SOC = {"R1_fraction": 0.35, "tau1": 4.0, "tau2": 250.0}

def resistance_r1r2(time_diff_sequence, r0_result):

    seq = time_diff_sequence.copy()
    seq["Zustand_clean"] = (seq["Zustand"].str.strip().str.replace(r"\*+$", "", regex=True))
    seq["Time_dt"] = pd.to_datetime(seq["Time"], utc=True, errors="coerce")

    seq = (
        seq
        .dropna(subset=["Time_dt", "Current", "Voltage"])
        .sort_values(["File", "Time_dt"])
        .reset_index(drop=True)
    )

    # 重新根据 Zustand 连续性划分片段，用来找完整 pulse 段和后续 PAUO 段
    seq["segment_id"] = (seq["File"].ne(seq["File"].shift()) | seq["Zustand_clean"].ne(seq["Zustand_clean"].shift())).cumsum()

    # 用 ID 来定位 pulse 后面的 PAUO：例如 42_17 的后续 pause 只取 42_18
    seq_id_parts = seq["ID"].astype(str).str.extract(r"^(.*)_(\d+)$")
    seq["id_prefix"] = seq_id_parts[0]
    seq["id_number"] = pd.to_numeric(seq_id_parts[1], errors="coerce")

    r0_table = r0_result.copy()
    r0_table["Time_dt"] = pd.to_datetime(r0_table["Time"], utc=True, errors="coerce")

    # 只保留 50% SOC：其余 SOC（90%/10%）在这一版 R1/R2 拟合中存在系统性边界钳制，
    # 拟合结果不可信，不再输出，R0 计算流程本身不受影响。
    r0_table["SOC_clean"] = r0_table["SOC"].astype(str).str.strip()
    r0_table = r0_table[r0_table["SOC_clean"].eq("50%")].copy()

    def is_pulse_state(s):
        return str(s).startswith(("CHA", "DCH"))

    def simulate_voltage(param, t_s, current_a, ocv_before, pulse_end_s, r0_ohm):
        r1_ohm, r2_ohm, tau1_s, tau2_s, ocv_after = param

        ocv_v = np.where(
            t_s <= pulse_end_s,
            ocv_before + (ocv_after - ocv_before) * (t_s / pulse_end_s),
            ocv_after
        )

        v1 = np.zeros(len(t_s), dtype=float)
        v2 = np.zeros(len(t_s), dtype=float)

        for k in range(1, len(t_s)):
            dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
            a1 = np.exp(-dt / tau1_s)
            a2 = np.exp(-dt / tau2_s)

            v1[k] = a1 * v1[k - 1] + r1_ohm * (1.0 - a1) * current_a[k]
            v2[k] = a2 * v2[k - 1] + r2_ohm * (1.0 - a2) * current_a[k]

        return ocv_v + current_a * r0_ohm + v1 + v2

    result_rows = []

    for _, row in r0_table.iterrows():
        file_name = row["File"]
        pulse_time = row["Time_dt"]
        r0_ohm = float(row["R0"]) / 1000.0

        file_seq = seq[seq["File"].eq(file_name)].copy()
        if file_seq.empty:
            continue

        # 找到 r0_result 对应的 pulse 采样点所在片段
        pulse_candidates = file_seq[(file_seq["Time_dt"].eq(pulse_time)) & (file_seq["Zustand"].map(is_pulse_state))]

        if pulse_candidates.empty:
                continue

        pulse_segment_id = pulse_candidates.iloc[0]["segment_id"]

        pulse_df = file_seq[(file_seq["segment_id"].eq(pulse_segment_id)) & (file_seq["Zustand"].map(is_pulse_state))].copy()
        pulse_df = pulse_df.sort_values("Time_dt").reset_index(drop=True)

        # 如果 pulse 段第一个点 Current == 0，则在 R1/R2 拟合中也去掉该点
        if len(pulse_df) >= 2 and pulse_df["Current"].iloc[0] == 0:
            pulse_df = pulse_df.iloc[1:].reset_index(drop=True)

        if len(pulse_df) < 5:
            continue

        pulse_start_time = pulse_df["Time_dt"].iloc[0]
        pulse_end_time = pulse_df["Time_dt"].iloc[-1]
        # 在每个 pulse 前找前一个 PAUO 末端点
        prev_pauo = file_seq[(file_seq["Time_dt"] < pulse_start_time) & (file_seq["Zustand_clean"].eq("PAUO"))].copy()

        if prev_pauo.empty:
            continue

        prev_pauo_point = prev_pauo.sort_values("Time_dt").iloc[[-1]].copy()

        id_parts = str(row.get("ID", "")).rsplit("_", 1)
        if len(id_parts) != 2:
            continue

        pulse_id_prefix = id_parts[0]
        pulse_id_number = pd.to_numeric(id_parts[1], errors="coerce")
        if pd.isna(pulse_id_number):
            continue

        next_pauo_df = file_seq[
            file_seq["Time_dt"].gt(pulse_end_time)
            & file_seq["Zustand_clean"].eq("PAUO")
            & file_seq["id_prefix"].eq(pulse_id_prefix)
            & file_seq["id_number"].eq(pulse_id_number + 1)
        ].copy()
        next_pauo_df = next_pauo_df.sort_values("Time_dt").reset_index(drop=True)

        if len(next_pauo_df) < 5:
            continue

        # 脉冲前 PAUO 1 个点 + 脉冲过程所有点 + 脉冲后 PAUO 段所有点（直至下一个脉冲）
        fit_df = pd.concat(
            [prev_pauo_point, pulse_df, next_pauo_df],
            ignore_index=True,
            sort=False
        ).sort_values("Time_dt").reset_index(drop=True)

        t_s = (fit_df["Time_dt"] - fit_df["Time_dt"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)

        current_a = fit_df["Current"].to_numpy(dtype=float)
        voltage_v = fit_df["Voltage"].to_numpy(dtype=float)

        ocv_before = float(prev_pauo_point["Voltage"].iloc[0])
        ocv_after_obs = float(next_pauo_df["Voltage"].iloc[-1])
        ocv_slack = 0.05

        # 从拟合窗口开始，到脉冲结束，一共过了多少秒
        pulse_end_s = (pulse_end_time - fit_df["Time_dt"].iloc[0]).total_seconds()
        pulse_end_s = max(float(pulse_end_s), 1e-9)

        pulse_mask = fit_df["Zustand"].map(is_pulse_state).to_numpy(dtype=bool)
        pauo_mask = fit_df["Zustand_clean"].eq("PAUO").to_numpy(dtype=bool)
        prev_point_mask = fit_df["Time_dt"].eq(prev_pauo_point["Time_dt"].iloc[0]).to_numpy(dtype=bool)

        # 前一个 PAUO 末端点只用于初始化，不参与误差计算
        fit_mask = ~prev_point_mask
        pulse_fit_mask = fit_mask & pulse_mask
        pauo_fit_mask = fit_mask & pauo_mask

        n_pulse = int(pulse_fit_mask.sum())
        n_pauo = int(pauo_fit_mask.sum())

        if n_pulse == 0 or n_pauo == 0:
            continue

        # 用时间跨度归一化权重，而不是用点数归一化
        pulse_duration_s = (pulse_df["Time_dt"].iloc[-1] - pulse_df["Time_dt"].iloc[0]).total_seconds()
        pauo_duration_s = (next_pauo_df["Time_dt"].iloc[-1] - next_pauo_df["Time_dt"].iloc[0]).total_seconds()

        i_pulse_abs = float(np.nanmedian(np.abs(pulse_df["Current"].to_numpy(dtype=float))))

        if not np.isfinite(i_pulse_abs) or i_pulse_abs <= 1e-9:
            continue

        is_dch = str(row.get("Zustand", "")).startswith("DCH")
        is_cha = str(row.get("Zustand", "")).startswith("CHA")

        # 设置 pulse 段和 PAUO 段在拟合中的总权重占比（50% SOC 统一使用 0.6/0.4）
        pulse_weight_total = 0.6
        pauo_weight_total = 0.4

        pulse_weight_per_s = pulse_weight_total / max(pulse_duration_s, 1.0)
        pauo_weight_per_s = pauo_weight_total / max(pauo_duration_s, 1.0)

        weights = np.zeros(len(fit_df), dtype=float)

        t_s_full = (fit_df["Time_dt"] - fit_df["Time_dt"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)

        for k in range(1, len(fit_df)):
            dt_k = max(t_s_full[k] - t_s_full[k - 1], 0.0)

            if pulse_fit_mask[k]:
                weights[k] = np.sqrt(pulse_weight_per_s * dt_k)
            elif pauo_fit_mask[k]:
                weights[k] = np.sqrt(pauo_weight_per_s * dt_k)

        # 初值：先用 pulse 末端电压粗略估计动态电阻；
        # 再按 50% SOC 的 R1/R2 分配比例和 tau 初值生成 x0。
        i_pulse_mean = float(np.nanmedian(np.abs(pulse_df["Current"].to_numpy(dtype=float))))
        v_pulse_tail = float(np.nanmedian(pulse_df["Voltage"].tail(min(5, len(pulse_df))).to_numpy(dtype=float)))
        r_dyn_guess = abs((ocv_after_obs - v_pulse_tail) / i_pulse_mean) - r0_ohm
        r_dyn_guess = float(np.clip(r_dyn_guess, 0.002, 0.08))

        r1_fraction = float(np.clip(RC2_INITIAL_GUESS_50SOC["R1_fraction"], 0.05, 0.95))
        r2_fraction = 1.0 - r1_fraction

        x0 = np.array([
            r1_fraction * r_dyn_guess,       # R1, Ohm
            r2_fraction * r_dyn_guess,       # R2, Ohm
            float(RC2_INITIAL_GUESS_50SOC["tau1"]),  # tau1, s
            float(RC2_INITIAL_GUESS_50SOC["tau2"]),  # tau2, s
            ocv_after_obs
        ], dtype=float)

        # bounds：50% SOC 专用（这是三个 SOC 里唯一没有边界钳制问题的一组，保持不变）
        ocv_lower = ocv_after_obs - ocv_slack
        ocv_upper = ocv_after_obs + ocv_slack

        if is_cha:
            ocv_lower = max(ocv_lower, ocv_before)
        elif is_dch:
            ocv_upper = min(ocv_upper, ocv_before)

        tau2_lower = 30.0 if (is_cha and i_pulse_abs < 2.0) else 60.0
        lower = np.array([1e-6, 0.003, 0.5, tau2_lower, ocv_lower], dtype=float)
        upper = np.array([0.08, 0.05, 25.0, 700.0, ocv_upper], dtype=float)

        # 根据当前 PAUO 段长度动态限制 tau2 上限, tau2 上限 = min(当前条件原本的 tau2 上限, 0.5 * PAUO 时长)
        tau2_upper_dyn = min(upper[3], 0.5 * pauo_duration_s)

        # 防止 PAUO 过短时 tau2 上限过低
        tau2_upper_dyn = max(tau2_upper_dyn, 100.0)

        # 保证 tau2 上界一定大于下界
        tau2_upper_dyn = max(tau2_upper_dyn, lower[3] + 1.0)

        upper[3] = tau2_upper_dyn

        x0 = np.clip(x0, lower + 1e-9, upper - 1e-9)

        def residual(param):
            voltage_hat = simulate_voltage(
                param,
                t_s,
                current_a,
                ocv_before,
                pulse_end_s,
                r0_ohm
            )
            return (voltage_hat - voltage_v) * weights

        try:
            fit = least_squares(
                residual,
                x0=x0,
                bounds=(lower, upper),
                max_nfev=5000
            )

            r1_ohm, r2_ohm, tau1_s, tau2_s, _ = fit.x

            voltage_fit = simulate_voltage(
                fit.x,
                t_s,
                current_a,
                ocv_before,
                pulse_end_s,
                r0_ohm
            )

            err_mv = (voltage_fit[fit_mask] - voltage_v[fit_mask]) * 1000.0
            rmse_mv = float(np.sqrt(np.mean(err_mv ** 2)))

            y_true = voltage_v[fit_mask]
            y_pred = voltage_fit[fit_mask]

            ss_res = np.sum((y_true - y_pred) ** 2)
            ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

            if ss_tot > 0:
                r2_score = float(1.0 - ss_res / ss_tot)
            else:
                r2_score = np.nan

            c1_f = tau1_s / r1_ohm
            c2_f = tau2_s / r2_ohm

            output_row = row.drop(labels=["Time_dt", "SOC_clean"], errors="ignore").to_dict()
            output_row.update({
                "R1": r1_ohm * 1000.0,
                "R2": r2_ohm * 1000.0,
                "tau1": tau1_s,
                "tau2": tau2_s,
                "C1": c1_f,
                "C2": c2_f,
                "RMSE_mV": rmse_mv,
                "R2_score": r2_score
            })
            result_rows.append(output_row)

        except Exception:
            output_row = row.drop(labels=["Time_dt", "SOC_clean"], errors="ignore").to_dict()
            output_row.update({
                "R1": np.nan,
                "R2": np.nan,
                "tau1": np.nan,
                "tau2": np.nan,
                "C1": np.nan,
                "C2": np.nan,
                "RMSE_mV": np.nan,
                "R2_score": np.nan
            })
            result_rows.append(output_row)

    return pd.DataFrame(result_rows)


r1r2_result = resistance_r1r2(time_diff_sequence, r0_result)
display(r1r2_result)

r2_summary = (
    r1r2_result
    .groupby("SOC", as_index=False)
    .agg({
        "RMSE_mV": "mean",
        "R2_score": "mean"
    })
)

display(r2_summary)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,...,R0_Quality,R0_Confidence,R1,R2,tau1,tau2,C1,C2,RMSE_mV,R2_score
0,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,18.330317,...,正常,完全可信（权重1.0）,6.001486,14.343984,6.241361,49.585826,1039.969250,3456.907420,0.099775,0.999743
1,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,18.422246,...,正常,完全可信（权重1.0）,8.406469,29.500769,8.141399,156.568112,968.468392,5307.255248,0.456055,0.998749
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.570000+00:00,2.996974,3.837063,CHA,12_47,CHA/3.0,18.499146,...,正常,完全可信（权重1.0）,7.548925,18.020410,7.222060,100.555090,956.700428,5580.066586,0.300631,0.999481
3,89.1,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM14_...,2024-12-01 21:15:13.900000+00:00,1.499604,3.807599,CHA,14_39,CHA/1.5,18.938095,...,正常,完全可信（权重1.0）,6.419404,14.133318,6.865313,52.964387,1069.462647,3747.484356,0.101188,0.999749
4,89.1,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM14_...,2024-12-01 22:15:57.160000+00:00,-2.999864,3.719390,DCH,14_43,DCH/-3.0,18.859834,...,正常,完全可信（权重1.0）,8.545497,30.195313,8.081307,158.405398,945.680161,5246.026000,0.460643,0.998910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,77.5,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM54_...,2026-01-04 20:41:40.550000+00:00,-2.998424,3.714164,DCH,54_44,DCH/-3.0,26.008643,...,正常,完全可信（权重1.0）,10.682247,35.039448,7.854213,172.191047,735.258501,4914.205504,0.570737,0.998526
68,77.5,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM54_...,2026-01-04 21:42:43.910000+00:00,2.998592,3.880980,CHA,54_48,CHA/3.0,26.407536,...,正常,完全可信（权重1.0）,9.340014,20.763505,7.484233,118.934637,801.308520,5728.061784,0.389314,0.999706
69,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 21:29:06.100000+00:00,1.499694,3.801039,CHA,9_40,CHA/1.5,17.729444,...,正常,完全可信（权重1.0）,6.055196,14.390342,6.120045,52.250452,1010.709644,3630.938764,0.098654,0.999768
70,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,17.918675,...,正常,完全可信（权重1.0）,8.253342,28.950094,8.208391,157.353341,994.553520,5435.330850,0.457638,0.998718


,SOC,RMSE_mV,R2_score
0,50%,0.312787,0.9994


In [9]:
soh_r_table = r1r2_result.copy()

# 提取 CHA / DCH 信息
soh_r_table["Pulse_type"] = (
    soh_r_table["Zustand"]
    .astype(str)
    .str.extract(r"^(CHA|DCH)", expand=False)
)

# 电流取绝对值，避免 DCH 显示成负数
soh_r_table["Current_abs"] = soh_r_table["Current"].abs()

# 生成图例标签，例如：50% CHA 3.0A
soh_r_table["Curve_label"] = (
    soh_r_table["SOC"].astype(str)
    + " "
    + soh_r_table["Pulse_type"].astype(str)
    + " "
    + soh_r_table["Current_abs"].round(1).astype(str)
    + "A"
)

# 排序
soh_r_table = soh_r_table.sort_values(["SOH", "SOC", "Pulse_type", "Current_abs"])

# 先显示宽表：每个 pulse 一行，R0/R1/R2 在同一行
display(soh_r_table[["SOH", "SOC", "Zustand", "Pulse_type", "Current", "Current_abs", "R0", "R1", "R2", "tau1", "tau2"]])

# 本版本只保留了 50% SOC 的 R1/R2 拟合，因此只画一张图（不再按 SOC 分 3 列）
soh_r_table["Pulse_label"] = (
    soh_r_table["Pulse_type"].astype(str)
    + " "
    + soh_r_table["Current_abs"].round(1).astype(str)
    + "A"
)

pulse_order = ["CHA 1.5A", "CHA 3.0A", "DCH 3.0A"]

# 固定颜色，方便和之前三分格版本的配色保持一致
color_map = {
    "CHA 1.5A": "#AB63FA",
    "CHA 3.0A": "#FFA15A",
    "DCH 3.0A": "#19D3F3",
}

for resistance_name in ["R0", "R1", "R2", "tau1", "tau2"]:

    fig = go.Figure()

    for pulse in pulse_order:
        d = soh_r_table[soh_r_table["Pulse_label"] == pulse].sort_values("SOH")

        if d.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=d["SOH"],
                y=d[resistance_name],
                mode="lines+markers",
                name=pulse,
                line=dict(color=color_map[pulse], width=2),
                marker=dict(color=color_map[pulse], size=6),
                connectgaps=True
            )
        )

    if resistance_name.startswith("tau"):
        yaxis_title = f"{resistance_name} (s)"
    else:
        yaxis_title = f"{resistance_name} (mOhm)"

    fig.update_layout(
        template="plotly_white",
        title=f"SOH vs {resistance_name} (50% SOC)",
        width=600,
        height=500,
        legend_title_text="pulse / current",
        xaxis_title="SOH (%)",
        yaxis_title=yaxis_title,
    )

    fig.update_xaxes(autorange="reversed")

    fig.show()


,SOH,SOC,Zustand,Pulse_type,Current,Current_abs,R0,R1,R2,tau1,tau2
66,77.5,50%,CHA,CHA,1.497895,1.497895,25.968892,7.775741,16.771610,6.157110,55.301340
68,77.5,50%,CHA,CHA,2.998592,2.998592,26.407536,9.340014,20.763505,7.484233,118.934637
67,77.5,50%,DCH,DCH,-2.998424,2.998424,26.008643,10.682247,35.039448,7.854213,172.191047
63,77.8,50%,CHA,CHA,1.499155,1.499155,26.105769,7.786481,16.380764,6.850359,58.000977
65,77.8,50%,CHA,CHA,2.993016,2.993016,26.167756,8.955748,20.744735,7.124948,110.969548
...,...,...,...,...,...,...,...,...,...,...,...
71,92.2,50%,CHA,CHA,2.999671,2.999671,17.858465,7.004689,17.780334,6.590864,88.484108
70,92.2,50%,DCH,DCH,-2.999504,2.999504,17.918675,8.253342,28.950094,8.208391,157.353341
57,96.2,50%,CHA,CHA,1.499694,1.499694,16.969802,4.568904,14.849032,4.184204,37.287298
59,96.2,50%,CHA,CHA,2.999671,2.999671,17.281165,5.953490,16.305844,5.370547,62.689071
